# <span style="color:cyan">**Estimating GARCH Models with MLE and Bayesian Methods**</span>

This notebook demonstrates how to estimate Generalised Autoregressive Conditional Heteroskedasticity (GARCH) models using two statistical approaches:

1. **Maximum Likelihood Estimation (MLE):**  
   This approach estimates parameters by maximising the likelihood of observing the given data under the specified model. It provides point estimates and standard errors for the parameters.

2. **Bayesian Estimation:**  
   Bayesian estimation incorporates prior beliefs about parameters and updates these beliefs using observed data. The posterior distribution is sampled using the Random-Walk Metropolis-Hastings algorithm, enabling uncertainty quantification beyond point estimates.

This notebook specifically focuses on the GARCH(1,1)-Student-t model. The model assumes:

- The conditional variance $h_t$ evolves as:

  \begin{equation}
   h_t = \omega + \alpha r_{t-1}^2 + \beta h_{t-1},
   \end{equation}
   
   
   where $r_t$ is the return at time $t$.

- The returns $r_t$ follow a Student-t distribution with $\nu$ degrees of freedom, which allows for fat tails.

---

## **Content**

- **Data Preprocessing:** Load S&P 500 returns data, compute log-returns, and plot them.
- **Maximum Likelihood Estimation (MLE):**
  - Define the likelihood function for the GARCH(1,1)-Student-t model.
  - Use optimization techniques to estimate parameters.
  - Compute standard errors using the numerical Hessian.
- **Bayesian Estimation:**
  - Implement the Metropolis-Hastings algorithm to sample from the posterior.
  - Visualise posterior distributions and convergence diagnostics.
- **Visualizations and Comparisons:**
  - Histograms of posterior distributions.
  - Trace plots for Bayesian diagnostics.
  - Comparison of MLE and Bayesian estimates.

---



##  <span style="color:cyan">**Prerequisites and Required Packages**</span>


This notebook requires several Python packages for data handling, optimisation, and visualisation. Before running the notebook, ensure the following package is installed and the following function is defined:

- `progressbar`: For progress tracking during computations. (source: https://stackoverflow.com/questions/3160699/python-progress-bar)
- `numdifftools`: For numerical computation of the Hessian matrix (used for estimating standard erroumdifftools


In [ ]:
!pip install numdifftools

#or in Anaconda prompt/mac terminal: pip install numdifftools 

In [ ]:
# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize, fmin_slsqp
from scipy.special import gammaln
from scipy.stats import multivariate_normal
import math
import sys
import time

# For numerical computation of Hessian (standard errors)
import numdifftools as nd


# Linear algebra utilities
from numpy.linalg import inv

def progressbar(it, prefix="", size=60, fill="█", out=sys.stdout):  # Customise fill
    count = len(it)
    start = time.time()  # Time estimate start
    
    def show(j):
        x = int(size * j / count)
        remaining = ((time.time() - start) / j) * (count - j) if j > 0 else 0        
        mins, sec = divmod(remaining, 60)
        time_str = f"{int(mins):02}:{sec:03.1f}"
        print(f"{prefix}[{fill * x}{('.' * (size - x))}] {j}/{count} Est. wait {time_str}", end='\r', file=out, flush=True)
    
    show(0.1)  # Avoid div/0
    for i, item in enumerate(it):
        yield item
        show(i + 1)
    print("\n", flush=True, file=out)
    print("Finished! 🎉😁")  # Add done message
    print("\n", flush=True, file=out)
    print("----------------------------")



##  <span style="color:cyan">**Data Loading and Preprocessing**</span>

In this section, we will:

1. Load the S&P 500 dataset, which contains daily price information for the index.
2. Compute **log-returns** to transform the price data into a format suitable for volatility modeling. Log-returns are preferred because they:
   - Are additive over time.
   - Approximate continuously compounded returns.
3. Select a subset of the data to serve as the **in-sample estimation window** for model training.

### **Steps:**
- Load the CSV file.
- Compute daily log-returns:  
  \begin{equation}
  r_t = 100 \cdot \left(\ln(P_t) - \ln(P_{t-1})\right)
  \end{equation}

  where $P_t$ is the price on day $t$.
- Select the first 1500 observations as the in-sample data.

The resulting log-returns will be stored in a global variable `g_vR`, which will be used by the likelihood function and other computations.


In [ ]:
# Load data and compute log-returns
# Define global variable for log-returns
global g_vR

sIn = 'SP500_1998-2007.csv'
df  = pd.read_csv(sIn, index_col = 'Date',parse_dates=True)
df.drop(['Open', 'High', 'Low', 'Close', 'Volume'],axis=1,inplace=True)
vPrice = df.values;
iNumPrices = len(vPrice);
vReturns = 100 * ( np.log(vPrice[1:iNumPrices]) - np.log(vPrice[0:(iNumPrices-1)]) );

# Select estimation window of in-sample observations:

iNumInSampleObservations = 1500;
g_vR = vReturns[0:iNumInSampleObservations];

# Plot log-returns for visualization
plt.figure(figsize=(7, 4))
plt.plot(g_vR, label="Log-Returns")
plt.title("Log-Returns of S&P 500")
plt.xlabel("Days")
plt.ylabel("Log-Returns (%)")
plt.legend()
plt.grid(True)
plt.show()



---

##  <span style="color:cyan">**Maximum Likelihood Estimation (MLE)**</span>


In this section, we estimate the parameters of the GARCH(1,1)-Student-t model using Maximum Likelihood Estimation (MLE). The goal is to find the set of parameters that maximsze the likelihood of the observed log-returns.

### **Steps:**
1. Define the negative log-likelihood function.
2. Set parameter constraints to ensure the GARCH model is well-defined:
   - $\omega > 0$, $\alpha > 0$, $\beta > 0$.
   - $\alpha + \beta < 1$ for stationarity.
   - $\nu > 2$ for the Student-t distribution to have finite variance.
3. Use the `fmin_slsqp` optimization method to minimsze the negative log-likelihood.
4. Compute standard errors of the parameter estimates using the numerical Hessian matrix.

### **Output:**
- Maximum likelihood estimates of $(\omega, \alpha, \beta, \nu)$.
- Standard errors for each parameter.


In [ ]:
def fMinusLogLikelihoodGARCHStudentT(vTheta): 
    
    dOmega = vTheta[0]
    dAlpha = vTheta[1]
    dBeta  = vTheta[2]
    dNu    = vTheta[3]
    
    # Compute the variances vH[t] in the GARCH model 
    # (starting with the sample variance):
    
    iT = len(g_vR)  
    vH = np.ones((iT,1)) 
    
    vH[0] = np.var(g_vR)
    
    for t in range(1,iT):
        vH[t] = dOmega + dAlpha*g_vR[t-1]**2 + dBeta*vH[t-1]
    
    # Compute vector of log(density) values:
    # Note: \ symbol for code that continues on multiple lines:
    
    vLogPdf = gammaln( (dNu+1)/2 ) - gammaln( dNu/2 )                \
              - 0.5*np.log( (dNu-2)*np.pi*vH )                       \
              - 0.5*(dNu+1)*np.log( 1 +  (g_vR**2)/((dNu-2)*vH) )      
              
    dMinusLogLikelihood = - np.sum(vLogPdf)
    
    return dMinusLogLikelihood




# Initial values for optimization:

dOmega_ini = 0.1
dAlpha_ini = 0.05
dBeta_ini  = 0.90
dNu_ini    = 6    
    
vTheta_ini = [dOmega_ini, dAlpha_ini, dBeta_ini, dNu_ini]


# Restrictions for optimization: 
vOmega_bnds   = (0, 1)
vAlpha_bnds   = (0, 1)
vBeta_bnds    = (0, 1)
vNu_bnds      = (2.1, 30)

vTheta_bnds = [vOmega_bnds, vAlpha_bnds, vBeta_bnds, vNu_bnds]


# Estimation: minimise fMinusLogLikeStudentT function using fmin_slsqp command:

vTheta_ML = fmin_slsqp(fMinusLogLikelihoodGARCHStudentT, vTheta_ini, bounds = vTheta_bnds)
                   
   
# Compute standard errors as sqrt of diagonal elements
# of estimated covariance matrix of ML estimator, 
# where estimated covariance matrix = -(Hessian of loglikelihood evaluated at MLE)^{-1}.
# Note: the function already gives -loglikelihood, here we just need
#       Hessian^{-1} instead of -Hessian^{-1}.

mHessianofMinusLogL = nd.Hessian(fMinusLogLikelihoodGARCHStudentT)(vTheta_ML)
mCovariance_MLE     = inv(mHessianofMinusLogL)
vStandardErrors     = np.sqrt(np.diag(mCovariance_MLE)) 


# Print results with 4 decimals:

np.set_printoptions(precision=4)

# Print log-likelihood
print(f"\nLog-Likelihood: {-fMinusLogLikelihoodGARCHStudentT(vTheta_ML):.4f}")

# Print a header for parameter estimates and standard errors
print("\nMaximum Likelihood Estimation Results")
print("-" * 40)

# Create a formatted table for results
parameters = ["Omega", "Alpha", "Beta", "Nu"]
results = zip(parameters, vTheta_ML, vStandardErrors)

print(f"{'Parameter':<10} {'Estimate':>10} {'Std. Error':>15}")
print("-" * 40)
for param, estimate, stderr in results:
    print(f"{param:<10} {estimate:>10.4f} {stderr:>15.4f}")



---

## <span style="color:cyan">**Bayesian Estimation: Random-Walk Metropolis-Hastings**</span>

In this section, we use the **Random-Walk Metropolis-Hastings** algorithm to sample from the posterior distribution of the parameters in the GARCH(1,1)-Student-t model. This Bayesian approach provides a complete distribution for each parameter, allowing us to quantify uncertainty beyond point estimates.

### **Key Steps:**

1. **Setup:**
   - Start with an initial draw ($\theta_0$) from the feasible parameter space, using the MLE estimates as the initial values.
   - Use the covariance matrix of the MLE estimates as the proposal distribution for generating candidate draws.

2. **Candidate Draw:**
   - For each iteration $i$, propose a new parameter vector $\theta^*$ from a multivariate normal distribution centered at the current draw $\theta_{i-1}$:
     
\begin{equation}
\theta^* \sim N(\theta_{i-1}, \Sigma)
\end{equation}
where $\Sigma$ is the covariance matrix of the MLE estimates.

3. **Acceptance Probability:**
   - Compute the acceptance probability:
     \begin{equation}
     \alpha = \min\left(1, \frac{L(\theta^*)}{L(\theta_{i-1})}\right)
     \end{equation}
     where $L(\cdot)$ is the likelihood function.

4. **Acceptance/Rejection:**
   - Accept $\theta^*$ with probability $\alpha$. If rejected, retain the previous draw $\theta_{i-1}$.

5. **Burn-In and Posterior Analysis:**
   - Discard the first 100 draws (burn-in period) to ensure the Markov chain has reached a stationary distribution.
   - Use the remaining draws to compute posterior means, standard deviations, and visualise the parameter distributions.

### **Output:**
- Posterior means and standard deviations for the parameters:
  - $\omega$ (long-run variance level).
  - $\alpha$ (reaction to recent squared returns).
  - $\beta$ (persistence of past variance).
  - $\nu$ (degrees of freedom of the Student-t distribution).
- Histograms of posterior distributions for each parameter.
- Acceptance percentage to assess the efficiency of the Metropolis-Hastings algorithm.


In [ ]:
# Random-Walk Metropolis-Hastings method:

print("\nRandom-Walk Metropolis-Hastings Method:\n")

iNdraws = 1100            # Total number of draws
iBurnIn = 100             # Number of burn-in draws
iNumAcceptance = 0        # Counter for accepted draws

# Matrix for storing accepted (and repeated) draws:
mTheta = np.zeros((iNdraws, len(vTheta_ML)))

# Feasible initial value: theta_0 = the ML estimator:
mTheta[0] = vTheta_ML

# Use covariance matrix of MLE estimates for proposal distribution:
proposal_covariance = mCovariance_MLE


for i in progressbar((range(1, iNdraws)), "" , size=35, fill="∝" ):
    # Candidate draw: theta ~ N(theta_{i-1}, proposal_covariance)
    vTheta_candidate = multivariate_normal.rvs(mTheta[i - 1], proposal_covariance)

    # Check if the candidate is within bounds
    if (
        vTheta_candidate[0] > 0 and vTheta_candidate[1] > 0 and vTheta_candidate[2] > 0
        and vTheta_candidate[3] > 2 and vTheta_candidate[3] < 30
    ):
        # Compute acceptance probability:
        dAccProb = min(
            np.exp(
                -fMinusLogLikelihoodGARCHStudentT(vTheta_candidate)
                + fMinusLogLikelihoodGARCHStudentT(mTheta[i - 1])
            ),
            1,
        )

        # Draw a uniform random variable for acceptance/rejection:
        dU = np.random.uniform(0, 1)

        if dU < dAccProb:
            mTheta[i] = vTheta_candidate
            iNumAcceptance += 1
        else:
            mTheta[i] = mTheta[i - 1]  # Retain the previous draw if rejected
    else:
        mTheta[i] = mTheta[i - 1]  # Retain the previous draw if out of bounds

    # Update the progress bar

# Compute acceptance percentage:
dAcceptancePercentage = 100 * iNumAcceptance / iNdraws
print(f"\nAcceptance Percentage: {dAcceptancePercentage:.2f}%")

# Estimate posterior mean & standard deviation as sample mean & std of draws after burn-in:
vTheta_posterior_mean = np.mean(mTheta[iBurnIn:], axis=0)
vTheta_posterior_stdev = np.std(mTheta[iBurnIn:], axis=0)

print("\nPosterior Means and Standard Deviations:\n")
np.set_printoptions(precision=4)

parameters = ["Omega", "Alpha", "Beta", "Nu"]
print(f"{'Parameter':<10} {'Mean':>10} {'Std. Dev.':>15}")
print("-" * 40)

for i, param in enumerate(parameters):
    print(f"{param:<10} {vTheta_posterior_mean[i]:>10.4f} {vTheta_posterior_stdev[i]:>15.4f}")


---

## <span style="color:cyan">**Posterior Distributions**</span> 


In [ ]:


# Histograms of posterior distributions (draws after burn-in):
for i, param in enumerate(parameters):
    plt.figure(figsize=(8, 5))
    plt.hist(mTheta[iBurnIn:, i], bins=30, alpha=0.75, label=f"Posterior of {param}")
    plt.title(f"Posterior Distribution of {param}")
    plt.xlabel(param)
    plt.ylabel("Frequency")
    #plt.legend()
    plt.grid(True)
    #plt.savefig(f"{param}.pdf") #remove the hashtag of plt.savefig to save the figure
    plt.show()

---